# 01 — Data Ingestion & Live NAV Fetch
**Bluestock MF Capstone | Day 1**

This notebook covers:
1. Project folder verification
2. Load all local CSV datasets with QA checks
3. Fetch live NAV from mfapi.in for 6 key schemes
4. Explore fund_master metadata
5. Validate AMFI code coverage
6. Data quality summary


In [ ]:
import os, json, time, logging
from pathlib import Path
import pandas as pd
import numpy as np
import requests

# ── Paths ──────────────────────────────────────────────────────────────────
ROOT     = Path.cwd().parent          # adjust if needed
RAW_DIR  = ROOT / "data" / "raw"
PROC_DIR = ROOT / "data" / "processed"
DB_DIR   = ROOT / "data" / "db"
for d in [RAW_DIR, PROC_DIR, DB_DIR]:
    d.mkdir(parents=True, exist_ok=True)

print("Project root:", ROOT)
print("Folders OK ✓")


## 1. Load Local CSV Datasets

In [ ]:
# Load all CSVs in data/raw/
datasets = {}
csv_files = sorted(RAW_DIR.glob("*.csv"))
print(f"Found {len(csv_files)} CSV file(s) in data/raw/\n")

for fpath in csv_files:
    try:
        df = pd.read_csv(fpath, low_memory=False)
        datasets[fpath.stem] = df
        print(f"{'='*55}")
        print(f"File   : {fpath.name}")
        print(f"Shape  : {df.shape[0]:,} rows × {df.shape[1]} cols")
        print(f"Dtypes :\n{df.dtypes.to_string()}")
        print(f"\nHead   :")
        display(df.head(3))

        # Anomaly flags
        null_pct = df.isnull().mean().mul(100).round(2)
        flagged  = null_pct[null_pct > 0]
        if not flagged.empty:
            print(f"\n⚠ Null % by column:\n{flagged.to_string()}")
        dups = df.duplicated().sum()
        if dups:
            print(f"⚠ Duplicate rows: {dups}")
        print()
    except Exception as e:
        print(f"ERROR loading {fpath.name}: {e}")

if not csv_files:
    print("No local CSVs yet — they will appear here after the team adds them to data/raw/")


## 2. Fetch Live NAV from mfapi.in

In [ ]:
MFAPI_BASE = "https://api.mfapi.in/mf"

KEY_SCHEMES = {
    "HDFC_Top100_Direct":  125497,
    "SBI_Bluechip":        119551,
    "ICICI_Bluechip":      120503,
    "Nippon_LargeCap":     118632,
    "Axis_Bluechip":       119092,
    "Kotak_Bluechip":      120841,
}

def fetch_nav(code, retries=3):
    url = f"{MFAPI_BASE}/{code}"
    for i in range(1, retries+1):
        try:
            r = requests.get(url, timeout=15)
            r.raise_for_status()
            return r.json()
        except Exception as e:
            print(f"  Attempt {i} failed: {e}")
            time.sleep(2*i)
    return None

def parse_nav(data, code):
    meta    = data.get("meta", {})
    records = data.get("data", [])
    df = pd.DataFrame(records).rename(columns={"date":"nav_date","nav":"nav_value"})
    df["nav_date"]         = pd.to_datetime(df["nav_date"], format="%d-%m-%Y")
    df["nav_value"]        = pd.to_numeric(df["nav_value"], errors="coerce")
    df["scheme_code"]      = code
    df["scheme_name"]      = meta.get("scheme_name","")
    df["scheme_category"]  = meta.get("scheme_category","")
    df["fund_house"]       = meta.get("fund_house","")
    # fill weekends/holidays
    df = df.set_index("nav_date")
    full_idx = pd.date_range(df.index.min(), df.index.max(), freq="D")
    df = df.reindex(full_idx).ffill().bfill()
    df.index.name = "nav_date"
    return df.reset_index().sort_values("nav_date")

all_nav = {}
print(f"{'Scheme':<30} {'Latest Date':<15} {'Latest NAV':>12}  {'Rows':>6}")
print("-"*65)

for label, code in KEY_SCHEMES.items():
    data = fetch_nav(code)
    if data:
        df = parse_nav(data, code)
        all_nav[label] = df
        csv_path = RAW_DIR / f"nav_{label}_{code}.csv"
        df.to_csv(csv_path, index=False)
        latest = df.iloc[-1]
        print(f"{label:<30} {str(latest.nav_date.date()):<15} {latest.nav_value:>12.4f}  {len(df):>6,}")
    else:
        print(f"{label:<30} FETCH FAILED")
    time.sleep(0.3)


## 3. Explore Fund Master Metadata

In [ ]:
# Build a mini fund master from fetched data
rows = []
for label, df in all_nav.items():
    if not df.empty:
        r = df.iloc[0]
        rows.append({
            "scheme_code":     r.scheme_code,
            "scheme_name":     r.scheme_name,
            "fund_house":      r.fund_house,
            "scheme_category": r.scheme_category,
            "nav_start":       df["nav_date"].min().date(),
            "nav_end":         df["nav_date"].max().date(),
            "total_rows":      len(df),
        })

fund_master_df = pd.DataFrame(rows)
print("── Fund Master (from API) ──────────────────────────────")
display(fund_master_df)

print("\nUnique fund houses   :", fund_master_df["fund_house"].unique().tolist())
print("Unique categories    :", fund_master_df["scheme_category"].unique().tolist())


## 4. Validate AMFI Code Coverage

In [ ]:
fetched_codes = set(fund_master_df["scheme_code"].astype(int))
requested_codes = set(KEY_SCHEMES.values())

missing = requested_codes - fetched_codes
extra   = fetched_codes - requested_codes

print(f"Requested schemes : {len(requested_codes)}")
print(f"Successfully fetched: {len(fetched_codes)}")
print(f"Missing codes     : {missing if missing else 'None ✓'}")
print(f"Extra codes       : {extra   if extra   else 'None'}")

# If fund_master CSV exists, cross-check
fm_path = RAW_DIR / "fund_master.csv"
if fm_path.exists():
    fm = pd.read_csv(fm_path)
    fm_codes = set(fm["scheme_code"].astype(int))
    nav_codes = set()
    for df in all_nav.values():
        nav_codes.update(df["scheme_code"].astype(int).unique())
    orphaned = fm_codes - nav_codes
    print(f"\nCodes in fund_master but missing NAV history: {len(orphaned)}")
    if orphaned:
        print("  Sample:", list(orphaned)[:10])
else:
    print("\nfund_master.csv not yet added to data/raw/ — skipping cross-validation.")


## 5. Merge NAV Master & Save

In [ ]:
if all_nav:
    master = pd.concat(all_nav.values(), ignore_index=True)
    master.sort_values(["scheme_code","nav_date"], inplace=True)
    master.drop_duplicates(subset=["scheme_code","nav_date"], keep="last", inplace=True)
    master.reset_index(drop=True, inplace=True)

    out = PROC_DIR / "nav_master.csv"
    master.to_csv(out, index=False)
    print(f"Saved nav_master.csv → {len(master):,} rows")
    display(master.head())
    display(master.dtypes.rename("dtype"))


## 6. Data Quality Summary

In [ ]:
print("DATA QUALITY REPORT")
print("="*55)

if all_nav:
    print(f"\nNAV Master")
    print(f"  Schemes       : {master['scheme_code'].nunique()}")
    print(f"  Total rows    : {len(master):,}")
    print(f"  Date range    : {master['nav_date'].min().date()} → {master['nav_date'].max().date()}")
    print(f"  Null nav_value: {master['nav_value'].isnull().sum()}")

    print("\nPer-scheme row counts:")
    counts = master.groupby("scheme_name")["nav_date"].count().sort_values(ascending=False)
    print(counts.to_string())

for name, df in datasets.items():
    print(f"\n{name}")
    print(f"  Shape        : {df.shape}")
    null_pct = df.isnull().mean().mul(100).round(1)
    bad = null_pct[null_pct > 0]
    if not bad.empty:
        print(f"  Nulls (%)    : {bad.to_dict()}")
    print(f"  Duplicates   : {df.duplicated().sum()}")

print("\n✓ Day 1 data ingestion complete.")
